In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as _sum, count, avg, row_number, desc
from pyspark.sql.window import Window

from pyspark.sql import SparkSession
spark = SparkSession.builder.appName("Tugas7_KopiNusantara").getOrCreate()

In [6]:
# Membaca data langsung dari folder lokal praktikum-bigdata
df_cabang = spark.read.json("tugas7_cabang.json")
df_menu = spark.read.csv("tugas7_menu.csv", header=True, inferSchema=True)
df_transaksi = spark.read.csv("tugas7_transaksi.csv", header=True, inferSchema=True)

# Tampilkan skema untuk verifikasi
df_cabang.printSchema()
df_menu.printSchema()
df_transaksi.printSchema()

root
 |-- cabang_id: long (nullable = true)
 |-- kepala_barista: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- nama_cabang: string (nullable = true)

root
 |-- menu_id: integer (nullable = true)
 |-- nama_menu: string (nullable = true)
 |-- kategori_menu: string (nullable = true)
 |-- harga: integer (nullable = true)

root
 |-- trx_id: string (nullable = true)
 |-- cabang_id: integer (nullable = true)
 |-- menu_id: integer (nullable = true)
 |-- qty: integer (nullable = true)



In [7]:
from pyspark.sql.functions import col

df_join = df_transaksi.join(df_cabang, on="cabang_id", how="left") \
                      .join(df_menu, on="menu_id", how="left")

df_join = df_join.withColumn("total_penjualan", col("qty") * col("harga"))
df_join.show(5)

+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|menu_id|cabang_id|trx_id|qty|kepala_barista|      kota|         nama_cabang|nama_menu| kategori_menu|harga|total_penjualan|
+-------+---------+------+---+--------------+----------+--------------------+---------+--------------+-----+---------------+
|     14|        1|   KN0|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-14|          Kopi|18000|          72000|
|     11|        1|   KN1|  4|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          88000|
|     11|        1|   KN2|  3|         Nadia|  Magelang|Kopi Nusantara Ma...|  Menu-11|          Kopi|22000|          66000|
|     11|        3|   KN3|  1|         Nadia|  Semarang|Kopi Nusantara Se...|  Menu-11|          Kopi|22000|          22000|
|     12|        2|   KN4|  4|          Reza|Yogyakarta|Kopi Nusantara Yo...|  Menu-12|Makanan Ringan|35000|         140000|


In [8]:
from pyspark.sql.window import Window
from pyspark.sql.functions import sum, row_number, desc

df_agg_menu = df_join.groupBy("nama_cabang", "nama_menu") \
                     .agg(sum("total_penjualan").alias("total_penjualan_menu"))

windowSpec = Window.partitionBy("nama_cabang").orderBy(desc("total_penjualan_menu"))

df_top2_menu = df_agg_menu.withColumn("rank", row_number().over(windowSpec)) \
                          .filter(col("rank") <= 2)

df_top2_menu.show()

+--------------------+---------+--------------------+----+
|         nama_cabang|nama_menu|total_penjualan_menu|rank|
+--------------------+---------+--------------------+----+
|Kopi Nusantara Ma...|   Menu-1|             8505000|   1|
|Kopi Nusantara Ma...|  Menu-12|             8085000|   2|
|Kopi Nusantara Se...|  Menu-13|             8750000|   1|
|Kopi Nusantara Se...|   Menu-9|             8715000|   2|
|Kopi Nusantara Yo...|  Menu-12|             9275000|   1|
|Kopi Nusantara Yo...|  Menu-15|             8160000|   2|
+--------------------+---------+--------------------+----+



In [9]:
df_join.createOrReplaceTempView("kopi_nusantara")

query = """
    SELECT kepala_barista, SUM(total_penjualan) as total_penjualan_barista
    FROM kopi_nusantara
    GROUP BY kepala_barista
    ORDER BY total_penjualan_barista DESC
"""

spark.sql(query).show()

+--------------+-----------------------+
|kepala_barista|total_penjualan_barista|
+--------------+-----------------------+
|         Nadia|              177680000|
|          Reza|               84645000|
+--------------+-----------------------+



In [11]:
output_path = "laporan_tugas7" # Disimpan di folder lokal

# Simpan data
df_join.write.partitionBy("kategori_menu").mode("overwrite").parquet(output_path)

# Verifikasi: Baca kembali data
df_verify = spark.read.parquet(output_path)
print(f"Jumlah baris data: {df_verify.count()}")

Jumlah baris data: 4000


In [12]:
from pyspark.sql.functions import sum, count, avg

# Membuat tabel ringkasan per cabang
df_ringkasan = df_join.groupBy("nama_cabang").agg(
    sum("total_penjualan").alias("total_penjualan"),
    count("trx_id").alias("jumlah_transaksi"),
    avg("total_penjualan").alias("rata_rata_transaksi")
).orderBy(col("total_penjualan").desc())

df_ringkasan.show(truncate=False)

+-------------------------+---------------+----------------+-------------------+
|nama_cabang              |total_penjualan|jumlah_transaksi|rata_rata_transaksi|
+-------------------------+---------------+----------------+-------------------+
|Kopi Nusantara Semarang  |90542000       |1393            |64997.8463747308   |
|Kopi Nusantara Magelang  |87138000       |1302            |66926.26728110599  |
|Kopi Nusantara Yogyakarta|84645000       |1305            |64862.06896551724  |
+-------------------------+---------------+----------------+-------------------+



### F. Ringkasan Eksekutif & Rekomendasi

Berdasarkan analisis data dari 4000 transaksi Kopi Nusantara yang telah diproses, kita dapat memetakan performa dari masing-masing cabang secara komprehensif. Cabang Kopi Nusantara Semarang berhasil mencatatkan diri sebagai cabang dengan kinerja terbaik. Cabang ini menghasilkan total penjualan tertinggi serta jumlah transaksi terbanyak dibandingkan cabang lainnya. Pencapaian ini menunjukkan bahwa strategi operasional, pelayanan barista, dan penawaran produk di cabang tersebut sudah berjalan dengan sangat efektif dan diterima baik oleh pelanggan.

Di sisi lain, cabang Kopi Nusantara Yogyakarta menunjukkan performa penjualan yang paling rendah dan masih membutuhkan strategi khusus untuk meningkatkan omset harian. Berdasarkan temuan analisis window function pada bagian C, kami merekomendasikan agar cabang terlemah ini lebih gencar mempromosikan Menu-12 dan Menu-13.

Alasan utama dari rekomendasi ini adalah karena kedua menu tersebut telah terbukti menjadi daya tarik utama dan penyumbang omset terbesar (top-seller) di cabang-cabang lain (seperti Yogyakarta, Magelang, dan Semarang). Dengan memfokuskan strategi pemasaran pada "menu juara" ini—misalnya melalui program promosi media sosial lokal, pemberian sampel gratis untuk pengunjung baru, atau paket bundling dengan harga khusus—cabang yang tertinggal dapat meminimalisir risiko eksperimen menu baru. Strategi cross-selling menggunakan produk yang sudah terbukti diminati pasar ini diharapkan mampu mendongkrak jumlah kunjungan pelanggan, meningkatkan rata-rata nilai transaksi (ticket size), dan secara bertahap memulihkan kinerja finansial cabang tersebut.